**Assignment 2: Spark DataFrames and Transformations**\
**25MML1001 Karthik M**

# Theory

Q. Differentiate between transformations and actions in Spark. Provide at least 3 examples for each and explain their significance in execution planning.



## Definition

**Transformations** are *lazy* operations applied on a DataFrame or RDD that define a computation plan but do **not execute immediately**. Spark records them as a DAG (Directed Acyclic Graph) and waits for an action to trigger actual execution.

**Actions** are *eager* operations that **trigger the execution** of all recorded transformations and either return a result to the driver program or write data to an output sink.

---

## Key Differences Between Transformations and Actions

| Feature | Transformations | Actions |
|---|---|---|
| Execution | Lazy – not executed immediately | Eager – executed immediately |
| Return type | Returns a new DataFrame / RDD | Returns a value, list, or writes to storage |
| Purpose | Define the computation plan | Trigger the computation plan |
| DAG effect | Adds a step to the DAG | Submits the DAG to the cluster for execution |
| Example | `filter()`, `select()`, `groupBy()` | `show()`, `count()`, `collect()` |
| Fault tolerance | Recomputed from lineage if a node fails | Re-triggers the full DAG from scratch |
| Optimization | Catalyst Optimizer can reorder/merge them | Cannot be optimized – must run as-is |
| Frequency of use | Called many times to build pipeline | Called sparingly – each call is expensive |
| Output | No output to user | Returns output to driver or external storage |
| Chaining | Can be chained endlessly | Cannot be chained (terminates the pipeline) |

---

## Transformations:

### 1. `filter()` / `where()`
Filters rows from the DataFrame based on a given condition. Only rows that satisfy the condition are kept in the resulting DataFrame. Since it is lazy, no data is actually scanned until an action is called.

### 2. `select()`
Projects specific columns from the DataFrame, discarding the rest. This feeds into **column pruning** during optimization — Spark avoids loading unused columns entirely, saving memory and I/O.

### 3. `groupBy()`
Groups rows that share the same value in one or more columns. Almost always followed by an aggregation function like `count()`, `avg()`, or `sum()`. The actual grouping and aggregation only happens when an action is triggered.

---

## Actions:

### 1. `show()`
Triggers execution and prints the top N rows (default 20) to the console. This is the most commonly used action during development and debugging.

### 2. `count()`
Triggers execution and returns the total number of rows as a Python integer. Even though it looks simple, it scans the entire dataset — so it can be expensive on large DataFrames.

### 3. `collect()`
Retrieves **all** rows from the distributed DataFrame and pulls them into the driver machine as a Python list of Row objects.  


---

## Significance in Execution Planning

1. **Catalyst Optimizer** – When an action is called, Spark's Catalyst Optimizer analyzes the entire DAG and rewrites the execution plan for maximum efficiency before running anything.

2. **Predicate Pushdown** – Spark automatically moves `filter()` operations as early as possible in the plan, reducing the amount of data processed in later stages.

3. **Column Pruning** – Unused columns identified from `select()` calls are dropped early, minimizing memory and I/O usage throughout the pipeline.

4. **Stage Pipelining** – Multiple transformations are combined into a single execution stage where possible, avoiding unnecessary data shuffles across the cluster.

5. **Avoiding Redundant Computation** – Because transformations are lazy, Spark can detect duplicate sub-plans in the DAG and compute them only once, reusing the result.

# Practical

Q. Using PySpark:
- Load a sample dataset (CSV or manually created)
- Perform the following transformations:
  - Select specific columns
  - Apply a filter condition
  - Group data based on a column and calculate count
- Display the results using appropriate actions

In [1]:
#Import libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, avg

In [2]:
#Initiate Spark Session
spark = SparkSession.builder \
    .appName("DA2") \
    .getOrCreate()

In [3]:
# Manually create a sample employee dataset
data = [
    (1, "Karthik", "Engineering", 45000),
    (2, "Priya",   "HR",          28000),
    (3, "Arjun",   "Engineering", 62000),
    (4, "Meena",   "Marketing",   31000),
    (5, "Ravi",    "Engineering", 75000),
    (6, "Divya",   "HR",          22000),
    (7, "Suresh",  "Marketing",   41000),
    (8, "Nisha",   "Engineering", 58000)
]

In [4]:
#Create DataFrame
columns = ["ID", "Name", "Department", "Salary"]
df = spark.createDataFrame(data, columns)

In [5]:
#Show all records
df.show()

+---+-------+-----------+------+
| ID|   Name| Department|Salary|
+---+-------+-----------+------+
|  1|Karthik|Engineering| 45000|
|  2|  Priya|         HR| 28000|
|  3|  Arjun|Engineering| 62000|
|  4|  Meena|  Marketing| 31000|
|  5|   Ravi|Engineering| 75000|
|  6|  Divya|         HR| 22000|
|  7| Suresh|  Marketing| 41000|
|  8|  Nisha|Engineering| 58000|
+---+-------+-----------+------+



In [6]:
# TRANSFORMATION 1: Select specific columns
print("Select specific columns:")
df_selected = df.select("Name", "Department")
df_selected.show()

Select specific columns:
+-------+-----------+
|   Name| Department|
+-------+-----------+
|Karthik|Engineering|
|  Priya|         HR|
|  Arjun|Engineering|
|  Meena|  Marketing|
|   Ravi|Engineering|
|  Divya|         HR|
| Suresh|  Marketing|
|  Nisha|Engineering|
+-------+-----------+



In [7]:
# TRANSFORMATION 2: Apply a filter condition
print("Filter = Salary more than 35000:")
df_filtered = df.filter(df["Salary"] > 35000)
df_filtered.show()

Filter = Salary more than 35000:
+---+-------+-----------+------+
| ID|   Name| Department|Salary|
+---+-------+-----------+------+
|  1|Karthik|Engineering| 45000|
|  3|  Arjun|Engineering| 62000|
|  5|   Ravi|Engineering| 75000|
|  7| Suresh|  Marketing| 41000|
|  8|  Nisha|Engineering| 58000|
+---+-------+-----------+------+



In [8]:
# TRANSFORMATION 3: Group data based on a column and calculate count
print("Group by Department to find count and avg. salary:")
df_grouped = df.groupBy("Department") \
               .agg(
                   count("ID").alias("Employee_Count"),
                   avg("Salary").alias("Avg_Salary")
               )
df_grouped.show()

Group by Department to find count and avg. salary:
+-----------+--------------+----------+
| Department|Employee_Count|Avg_Salary|
+-----------+--------------+----------+
|Engineering|             4|   60000.0|
|         HR|             2|   25000.0|
|  Marketing|             2|   36000.0|
+-----------+--------------+----------+



In [9]:
#End Spark Session
spark.stop()